# Multivariate Time-Series Forecasting

## 2. Baseline Model

This notebook establishes baseline forecasting performance before applying deep learning models.

The baseline provides a reference point for evaluating the performance of:

- RNN
- LSTM
- GRU
- LSTM with Attention
- Transformer

The final engineered dataset will be used as the input source.

Because the dataset contains millions of observations, the data will be processed using memory-efficient Parquet operations rather than loading the complete dataset into memory.

The baseline evaluation will use a time-based train-validation split to preserve the temporal ordering of the data and prevent future information from being used during training.

Model performance will be evaluated using appropriate forecasting metrics and will be compared with the performance of subsequent deep learning models.

In [1]:
import gc
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

print("Libraries imported successfully.")

Libraries imported successfully.


In [14]:
import os

print("MyDrive path:")
print("/content/drive/MyDrive")

print("\nTop-level folders/files:")
for item in os.listdir("/content/drive/MyDrive"):
    print(item)

MyDrive path:
/content/drive/MyDrive

Top-level folders/files:
Colab Notebooks
20240322_184337.jpg
Sanjay Arora Bansi Lal.pdf
Emailing Jam Ma Real analysis pyq.pdf
tifr Linear Algebra Pyq.pdf
Jam Ma Real analysis pyq.pdf
NBHM Linear Algebra Pyq.pdf
Csir net Linear Algebra Pyq.pdf
Gate Ma Linear Algebra pyq.pdf
Jam Ma Linear Algebra Pyq.pdf
EWS form (1).pdf
Screenshot_20240322-181320_Chrome.jpg
M121M89AdmitCard (1) (2) (1).pdf
20240322_190639.jpg
AUD-20231007-WA0038.m4a
AUD-20231010-WA0060.mp3
AUD-20231010-WA0075.mp3
AUD-20231011-WA0086.mp3
gitika 1.mp3
Call recording +917023252073_221029_181115.m4a
Call recording +917023252073_221029_181455.m4a
Call recording +918955517480_230827_110831.m4a
Call recording +918955517480_230827_110839.m4a
Call recording +918955517480_230827_110949.m4a
Call recording +918955517480_230827_111305.m4a
Call recording 9513392208_240331_141850.m4a
Call recording amita jaipur_230423_132825.m4a
Call recording amita jaipur_230423_133023.m4a
Call recording amita ja

In [11]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import pyarrow.parquet as pq

data_path = "/content/drive/MyDrive/features_event_snap.parquet"

parquet_file = pq.ParquetFile(data_path)

print("====================================")
print("Final Feature Dataset")
print("====================================")

print("\nRows:")
print(f"{parquet_file.metadata.num_rows:,}")

print("\nColumns:")
print(len(parquet_file.schema.names))

print("\nNumber of row groups:")
print(parquet_file.num_row_groups)

print("\nColumn names:")
print(parquet_file.schema.names)

Final Feature Dataset

Rows:
58,327,370

Columns:
42

Number of row groups:
61

Column names:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available', 'day_of_month', 'week_of_year', 'day_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'price_change_1', 'price_change_pct_1', 'price_relative_7', 'is_event_day', 'event_count', 'snap_active']


In [17]:
# Check the temporal coverage of the final feature dataset

date_column = "date"

sample_table = parquet_file.read_row_group(
    0,
    columns=[date_column]
).to_pandas()

print("====================================")
print("Date Coverage Check")
print("====================================")

print("\nFirst date in sample row group:")
print(sample_table[date_column].min())

print("\nLast date in sample row group:")
print(sample_table[date_column].max())

del sample_table
gc.collect()

Date Coverage Check

First date in sample row group:
2011-01-29 00:00:00

Last date in sample row group:
2013-12-12 00:00:00


78

In [18]:
# Determine the global date range using Parquet metadata

date_column_index = parquet_file.schema.names.index("date")

min_dates = []
max_dates = []

for i in range(parquet_file.num_row_groups):
    column_metadata = parquet_file.metadata.row_group(i).column(date_column_index)
    statistics = column_metadata.statistics

    if statistics is not None:
        if statistics.min is not None:
            min_dates.append(pd.to_datetime(statistics.min))
        if statistics.max is not None:
            max_dates.append(pd.to_datetime(statistics.max))

global_min_date = min(min_dates)
global_max_date = max(max_dates)

print("====================================")
print("Global Date Coverage")
print("====================================")

print("\nFirst date:")
print(global_min_date)

print("\nLast date:")
print(global_max_date)

print("\nTotal calendar days:")
print((global_max_date - global_min_date).days + 1)

Global Date Coverage

First date:
2011-01-29 00:00:00

Last date:
2016-04-24 00:00:00

Total calendar days:
1913


In [19]:
# Define a time-based train-validation split

VALIDATION_DAYS = 28

train_end_date = global_max_date - pd.Timedelta(days=VALIDATION_DAYS)
validation_start_date = train_end_date + pd.Timedelta(days=1)

print("====================================")
print("Train-Validation Split")
print("====================================")

print("\nTraining period:")
print(f"{global_min_date.date()} → {train_end_date.date()}")

print("\nValidation period:")
print(f"{validation_start_date.date()} → {global_max_date.date()}")

print("\nTraining days:")
print((train_end_date - global_min_date).days + 1)

print("\nValidation days:")
print((global_max_date - validation_start_date).days + 1)

Train-Validation Split

Training period:
2011-01-29 → 2016-03-27

Validation period:
2016-03-28 → 2016-04-24

Training days:
1885

Validation days:
28


## 3. Baseline Forecasting Strategies

Three baseline forecasting strategies will be evaluated:

### 1. Naive Baseline

The forecast for a given day is the sales observed on the previous day.

\[
\hat{y}_t = y_{t-1}
\]

This provides a simple reference point.

### 2. Seasonal Naive Baseline

The forecast for a given day is the sales observed on the same day of the previous week.

\[
\hat{y}_t = y_{t-7}
\]

This is the primary baseline because the EDA showed a strong weekly sales pattern.

### 3. 28-Day Moving Average

The forecast is based on the average sales observed during the previous 28 days.

\[
\hat{y}_t =
\frac{1}{28}
\sum_{i=1}^{28} y_{t-i}
\]

The current day's sales are excluded from all baseline calculations to prevent data leakage.

All baselines will be evaluated on the same 28-day validation period.

In [20]:
# Columns required for baseline evaluation

baseline_columns = [
    "item_id",
    "store_id",
    "date",
    "sales",
    "lag_1",
    "lag_7",
    "rolling_mean_28"
]

print("Baseline columns:")
for col in baseline_columns:
    print("-", col)

Baseline columns:
- item_id
- store_id
- date
- sales
- lag_1
- lag_7
- rolling_mean_28


In [21]:
# Load only the validation period and required baseline columns

validation_columns = [
    "item_id",
    "store_id",
    "date",
    "sales",
    "lag_1",
    "lag_7",
    "rolling_mean_28"
]

validation_data = []

for i in range(parquet_file.num_row_groups):
    df = parquet_file.read_row_group(
        i,
        columns=validation_columns
    ).to_pandas()

    df["date"] = pd.to_datetime(df["date"])

    df = df[
        (df["date"] >= validation_start_date) &
        (df["date"] <= global_max_date)
    ]

    if len(df) > 0:
        validation_data.append(df)

    del df

    if (i + 1) % 10 == 0 or i == parquet_file.num_row_groups - 1:
        print(f"Processed row groups: {i + 1}/{parquet_file.num_row_groups}")

validation_df = pd.concat(validation_data, ignore_index=True)

del validation_data
gc.collect()

print("\n====================================")
print("Validation Dataset")
print("====================================")

print("\nRows:")
print(f"{len(validation_df):,}")

print("\nColumns:")
print(validation_df.columns.tolist())

print("\nDate range:")
print(validation_df["date"].min(), "→", validation_df["date"].max())

print("\nUnique item-store series:")
print(validation_df[["item_id", "store_id"]].drop_duplicates().shape[0])

Processed row groups: 10/61
Processed row groups: 20/61
Processed row groups: 30/61
Processed row groups: 40/61
Processed row groups: 50/61
Processed row groups: 60/61
Processed row groups: 61/61

Validation Dataset

Rows:
853,720

Columns:
['item_id', 'store_id', 'date', 'sales', 'lag_1', 'lag_7', 'rolling_mean_28']

Date range:
2016-03-28 00:00:00 → 2016-04-24 00:00:00

Unique item-store series:
30490


In [22]:
# Generate predictions for all baseline strategies

validation_df["naive_pred"] = validation_df["lag_1"]
validation_df["seasonal_naive_pred"] = validation_df["lag_7"]
validation_df["moving_average_28_pred"] = validation_df["rolling_mean_28"]

print("====================================")
print("Baseline Predictions")
print("====================================")

prediction_columns = [
    "sales",
    "naive_pred",
    "seasonal_naive_pred",
    "moving_average_28_pred"
]

print("\nPrediction columns created:")
for col in prediction_columns:
    print("-", col)

print("\nMissing predictions:")
print(
    validation_df[
        ["naive_pred", "seasonal_naive_pred", "moving_average_28_pred"]
    ].isna().sum()
)

Baseline Predictions

Prediction columns created:
- sales
- naive_pred
- seasonal_naive_pred
- moving_average_28_pred

Missing predictions:
naive_pred                0
seasonal_naive_pred       0
moving_average_28_pred    0
dtype: int64


In [23]:
# Evaluate baseline forecasting performance

def calculate_mae(actual, predicted):
    return np.mean(np.abs(actual - predicted))


def calculate_rmse(actual, predicted):
    return np.sqrt(np.mean((actual - predicted) ** 2))


def calculate_wape(actual, predicted):
    denominator = np.sum(np.abs(actual))

    if denominator == 0:
        return np.nan

    return np.sum(np.abs(actual - predicted)) / denominator


baseline_results = []

models = {
    "Naive": "naive_pred",
    "Seasonal Naive (7-day)": "seasonal_naive_pred",
    "28-day Moving Average": "moving_average_28_pred"
}

for model_name, prediction_column in models.items():

    actual = validation_df["sales"].to_numpy()
    predicted = validation_df[prediction_column].to_numpy()

    mae = calculate_mae(actual, predicted)
    rmse = calculate_rmse(actual, predicted)
    wape = calculate_wape(actual, predicted)

    baseline_results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "WAPE": wape
    })

baseline_results_df = pd.DataFrame(baseline_results)

print("====================================")
print("Baseline Model Performance")
print("====================================")

print(
    baseline_results_df.to_string(
        index=False,
        formatters={
            "MAE": "{:.4f}".format,
            "RMSE": "{:.4f}".format,
            "WAPE": "{:.4%}".format
        }
    )
)

Baseline Model Performance
                 Model    MAE   RMSE     WAPE
                 Naive 1.1798 2.5948 85.0972%
Seasonal Naive (7-day) 1.2054 2.6601 86.9390%
 28-day Moving Average 1.0050 2.0935 72.4875%


In [24]:
# Evaluate baseline performance by store

store_results = []

for store, group in validation_df.groupby("store_id"):

    actual = group["sales"].to_numpy()

    for model_name, prediction_column in models.items():

        predicted = group[prediction_column].to_numpy()

        store_results.append({
            "store_id": store,
            "Model": model_name,
            "MAE": calculate_mae(actual, predicted),
            "RMSE": calculate_rmse(actual, predicted),
            "WAPE": calculate_wape(actual, predicted)
        })

store_results_df = pd.DataFrame(store_results)

print("====================================")
print("Store-level Baseline Performance")
print("====================================")

print(
    store_results_df
    .sort_values(["store_id", "WAPE"])
    .to_string(
        index=False,
        formatters={
            "MAE": "{:.4f}".format,
            "RMSE": "{:.4f}".format,
            "WAPE": "{:.4%}".format
        }
    )
)

Store-level Baseline Performance
store_id                  Model    MAE   RMSE      WAPE
    CA_1  28-day Moving Average 1.0876 2.1934  72.5618%
    CA_1                  Naive 1.2817 2.7682  85.5176%
    CA_1 Seasonal Naive (7-day) 1.3097 2.8066  87.3823%
    CA_2  28-day Moving Average 1.0972 2.0181  74.9175%
    CA_2 Seasonal Naive (7-day) 1.3166 2.5744  89.8953%
    CA_2                  Naive 1.3280 2.5845  90.6735%
    CA_3  28-day Moving Average 1.3398 2.7083  64.6310%
    CA_3                  Naive 1.5947 3.4150  76.9302%
    CA_3 Seasonal Naive (7-day) 1.6214 3.4156  78.2158%
    CA_4  28-day Moving Average 0.7615 1.3692  87.5952%
    CA_4                  Naive 0.9089 1.8458 104.5531%
    CA_4 Seasonal Naive (7-day) 0.9147 1.8588 105.2255%
    TX_1  28-day Moving Average 0.8235 1.6926  75.7739%
    TX_1                  Naive 0.9657 2.1853  88.8635%
    TX_1 Seasonal Naive (7-day) 0.9805 2.1920  90.2238%
    TX_2  28-day Moving Average 0.9407 1.8869  72.9350%
    TX_2       

In [25]:
# Create final baseline summary

overall_summary = baseline_results_df.copy()

best_model = overall_summary.loc[
    overall_summary["WAPE"].idxmin(), "Model"
]

naive_wape = overall_summary.loc[
    overall_summary["Model"] == "Naive", "WAPE"
].iloc[0]

best_wape = overall_summary.loc[
    overall_summary["Model"] == best_model, "WAPE"
].iloc[0]

improvement = (naive_wape - best_wape) / naive_wape * 100

print("====================================")
print("Final Baseline Summary")
print("====================================")

print("\nOverall Performance:")
print(
    overall_summary.to_string(
        index=False,
        formatters={
            "MAE": "{:.4f}".format,
            "RMSE": "{:.4f}".format,
            "WAPE": "{:.4%}".format
        }
    )
)

print("\nBest Baseline:")
print(best_model)

print("\nWAPE improvement over Naive:")
print(f"{improvement:.2f}%")

Final Baseline Summary

Overall Performance:
                 Model    MAE   RMSE     WAPE
                 Naive 1.1798 2.5948 85.0972%
Seasonal Naive (7-day) 1.2054 2.6601 86.9390%
 28-day Moving Average 1.0050 2.0935 72.4875%

Best Baseline:
28-day Moving Average

WAPE improvement over Naive:
14.82%


## 4. Baseline Results and Findings

The baseline models were evaluated using a time-based validation period covering the final 28 days of the available training data.

### Validation Period

- Training period: 2011-01-29 to 2016-03-27
- Validation period: 2016-03-28 to 2016-04-24
- Validation observations: 853,720
- Item-store series: 30,490

### Baseline Performance

| Model | MAE | RMSE | WAPE |
|---|---:|---:|---:|
| Naive | 1.1798 | 2.5948 | 85.10% |
| Seasonal Naive (7-day) | 1.2054 | 2.6601 | 86.94% |
| 28-day Moving Average | **1.0050** | **2.0935** | **72.49%** |

### Key Findings

The 28-day Moving Average achieved the best performance across all three evaluation metrics.

Compared with the Naive baseline, the 28-day Moving Average reduced WAPE by approximately **14.82%**.

The Seasonal Naive baseline did not outperform the Naive baseline during the validation period, despite the strong weekly pattern observed during EDA.

The results indicate that smoothing recent demand over a longer 28-day window provides a stronger baseline signal than relying only on the previous day or the same day of the previous week.

The 28-day Moving Average will therefore serve as the primary baseline benchmark for subsequent deep learning models.

Future models will be considered successful only if they improve upon this baseline under the same validation framework.

## 5. Conclusion

The baseline forecasting stage has established a reproducible benchmark for the multivariate time-series forecasting problem.

A time-based validation strategy was used to preserve temporal ordering and prevent future information from entering the baseline predictions.

Among the evaluated methods, the 28-day Moving Average achieved the strongest validation performance:

- MAE: **1.0050**
- RMSE: **2.0935**
- WAPE: **72.49%**

This baseline will be used as the reference point for evaluating the RNN, LSTM, GRU, LSTM with Attention, and Transformer models.

The baseline stage is complete.